# Financial News Sentiment Analysis Pipeline

This notebook implements a streamlined sentiment analysis system for financial assets using Google News scraping and FinBERT sentiment analysis.

## Process Overview:
1. Define assets and their search terms
2. For each asset and month (2015-2024):
   - Generate Google News URLs with date filters
   - Scrape first 10 articles for each term
   - Extract article content
   - Compute sentiment using FinBERT
   - Calculate asset sentiment score: (P_positive - P_negative) / N
3. Store monthly sentiment scores
4. Generate unified sentiment matrix


In [1]:
# Install required packages
%pip install transformers torch pandas beautifulsoup4 requests matplotlib seaborn


Note: you may need to restart the kernel to use updated packages.


In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime
import os
import warnings
warnings.filterwarnings('ignore')

# Import our utility functions
from sentiment_scraper_utils import SentimentScraper, get_asset_definitions

# Set random seed for reproducibility
np.random.seed(42)

print("Libraries imported successfully")


Libraries imported successfully


## Asset Configuration

Define the financial assets and their associated search terms for comprehensive news coverage.


In [3]:
# Load asset definitions
assets = get_asset_definitions()

# Display asset configuration
print("Asset Search Terms Configuration:")
print("=" * 40)
for asset, terms in assets.items():
    print(f"{asset}: {len(terms)} terms")
    print(f"  Sample terms: {terms[:3]}")
    print()


Asset Search Terms Configuration:
SP_500: 5 terms
  Sample terms: ['S&P 500', 'SP 500', 'SPX']

NASDAQ: 4 terms
  Sample terms: ['NASDAQ', 'NASDAQ Composite', 'NASDAQ Index']

Dow_Jones: 3 terms
  Sample terms: ['Dow Jones', 'DJIA', 'Dow Jones Industrial Average']

CAC_40: 3 terms
  Sample terms: ['CAC 40', 'Paris Stock Exchange', 'French Stock Market']

FTSE_100: 3 terms
  Sample terms: ['FTSE 100', 'London Stock Exchange', 'UK Stock Market']

EuroStoxx_50: 3 terms
  Sample terms: ['EuroStoxx 50', 'European Stock Market', 'Eurozone Stocks']

Nikkei_225: 3 terms
  Sample terms: ['Nikkei 225', 'Japanese Stock Market', 'Tokyo Stock Exchange']

Hang_Seng: 3 terms
  Sample terms: ['Hang Seng', 'Hong Kong Stock Market', 'HSI']

Shanghai_Composite: 3 terms
  Sample terms: ['Shanghai Composite', 'Chinese Stock Market', 'Shanghai Index']

Gold: 4 terms
  Sample terms: ['Gold', 'Gold Price', 'Gold Market']

Silver: 3 terms
  Sample terms: ['Silver', 'Silver Price', 'Silver Market']

Oil: 5 term

## Initialize Sentiment Scraper

Set up the sentiment analysis pipeline with FinBERT model.


In [ ]:
# Initialize the sentiment scraper
scraper = SentimentScraper(output_dir="sentiment_results")

# Load FinBERT model
scraper.load_finbert()

print("Sentiment scraper initialized successfully")


## Test Sentiment Analysis

Quick test of the sentiment analysis functionality before running the full pipeline.


In [ ]:
# Test sentiment analysis with sample texts
test_texts = [
    "The S&P 500 reached record highs as investors showed strong confidence in the market.",
    "Stock markets plummeted today amid fears of economic recession and rising inflation.",
    "The Federal Reserve announced its decision to maintain current interest rates."
]

print("Sentiment Analysis Test:")
print("=" * 50)
for i, text in enumerate(test_texts, 1):
    label, score = scraper.analyze_sentiment(text)
    print(f"Text {i}: {label.upper()} (score: {score:.3f})")
    print(f"Content: {text[:80]}...")
    print()


## Sample Monthly Scraping

Run sentiment analysis for a sample month to test the complete pipeline before processing all years.


In [ ]:
# Test with a single asset and month first
test_asset = "SP_500"
test_year = 2024
test_month = 1

print(f"Testing sentiment scraping for {test_asset} in {test_month:02d}/{test_year}")
print("=" * 60)

# Run sentiment analysis for the test case
sentiment_score = scraper.scrape_monthly_sentiment(
    asset_name=test_asset,
    terms=assets[test_asset][:2],  # Use only first 2 terms for testing
    year=test_year,
    month=test_month
)

print(f"\nFinal sentiment score for {test_asset}: {sentiment_score:.4f}")


## Full Pipeline Execution

Process all assets for selected months. This implementation focuses on recent years and key assets to demonstrate the complete workflow efficiently.


In [ ]:
# Configuration for full pipeline - start with recent years and key assets
START_YEAR = 2023
END_YEAR = 2024
SELECTED_ASSETS = ["SP_500", "NASDAQ", "Gold"]  # Key representative assets

print(f"Running pipeline for {len(SELECTED_ASSETS)} assets from {START_YEAR}-{END_YEAR}")
print("Selected assets:", SELECTED_ASSETS)
print("=" * 70)

# Store all results
all_results = []

for year in range(START_YEAR, END_YEAR + 1):
    for month in range(1, 13):
        print(f"\nProcessing {month:02d}/{year}")
        
        monthly_results = {}
        
        for asset_name in SELECTED_ASSETS:
            try:
                sentiment_score = scraper.scrape_monthly_sentiment(
                    asset_name=asset_name,
                    terms=assets[asset_name][:3],  # Use first 3 terms for efficiency
                    year=year,
                    month=month
                )
                monthly_results[asset_name] = sentiment_score
                
                # Store for unified matrix
                all_results.append({
                    'Year': year,
                    'Month': month,
                    'Asset': asset_name,
                    'Sentiment_Score': sentiment_score
                })
                
            except Exception as e:
                print(f"Error processing {asset_name}: {e}")
                monthly_results[asset_name] = 0.0
        
        # Save monthly results
        scraper.save_monthly_results(monthly_results, year, month)
        
        print(f"Completed {month:02d}/{year}")

print("\nPipeline execution completed!")


## Generate Unified Sentiment Matrix

Create a comprehensive matrix showing sentiment scores for all assets across all processed months.


In [ ]:
# Create unified sentiment matrix
if all_results:
    results_df = pd.DataFrame(all_results)
    
    # Create date column for better visualization
    results_df['Date'] = pd.to_datetime(results_df[['Year', 'Month']].assign(day=1))
    
    # Pivot to create matrix format
    sentiment_matrix = results_df.pivot_table(
        index='Date', 
        columns='Asset', 
        values='Sentiment_Score',
        fill_value=0
    )
    
    # Save unified matrix
    output_path = os.path.join(scraper.output_dir, "unified_sentiment_matrix.csv")
    sentiment_matrix.to_csv(output_path)
    
    print(f"Unified sentiment matrix saved to: {output_path}")
    print(f"Matrix shape: {sentiment_matrix.shape}")
    print("\nMatrix preview:")
    print(sentiment_matrix.head())
else:
    print("No results available to create unified matrix")


## Sentiment Analysis Visualization

Create visualizations to understand sentiment trends across assets and time periods.


In [ ]:
if 'sentiment_matrix' in locals():
    # Set up the plotting style
    plt.style.use('default')
    fig, axes = plt.subplots(2, 2, figsize=(15, 10))
    fig.suptitle('Financial Asset Sentiment Analysis', fontsize=16, fontweight='bold')
    
    # 1. Time series plot of sentiment scores
    ax1 = axes[0, 0]
    for asset in sentiment_matrix.columns:
        ax1.plot(sentiment_matrix.index, sentiment_matrix[asset], label=asset, linewidth=2)
    ax1.set_title('Sentiment Scores Over Time')
    ax1.set_xlabel('Date')
    ax1.set_ylabel('Sentiment Score')
    ax1.legend()
    ax1.grid(True, alpha=0.3)
    
    # 2. Heatmap of sentiment correlations
    ax2 = axes[0, 1]
    correlation_matrix = sentiment_matrix.corr()
    sns.heatmap(correlation_matrix, annot=True, cmap='RdBu_r', center=0, 
                square=True, ax=ax2, fmt='.2f')
    ax2.set_title('Asset Sentiment Correlations')
    
    # 3. Distribution of sentiment scores by asset
    ax3 = axes[1, 0]
    sentiment_data = [sentiment_matrix[asset].values for asset in sentiment_matrix.columns]
    asset_names = list(sentiment_matrix.columns)
    
    ax3.boxplot(sentiment_data, labels=asset_names)
    ax3.set_title('Sentiment Score Distributions')
    ax3.set_ylabel('Sentiment Score')
    ax3.tick_params(axis='x', rotation=45)
    
    # 4. Average sentiment by asset
    ax4 = axes[1, 1]
    avg_sentiment = sentiment_matrix.mean()
    bars = ax4.bar(avg_sentiment.index, avg_sentiment.values)
    ax4.set_title('Average Sentiment by Asset')
    ax4.set_ylabel('Average Sentiment Score')
    ax4.tick_params(axis='x', rotation=45)
    
    # Color bars based on sentiment
    for bar, value in zip(bars, avg_sentiment.values):
        if value > 0:
            bar.set_color('green')
        elif value < 0:
            bar.set_color('red')
        else:
            bar.set_color('gray')
    
    plt.tight_layout()
    
    # Save the visualization
    plot_path = os.path.join(scraper.output_dir, "sentiment_analysis_visualization.png")
    plt.savefig(plot_path, dpi=300, bbox_inches='tight')
    plt.show()
    
    print(f"Visualization saved to: {plot_path}")
else:
    print("No sentiment data available for visualization")


## Summary Statistics

Generate summary statistics and insights from the sentiment analysis results.


In [ ]:
if 'sentiment_matrix' in locals():
    print("SENTIMENT ANALYSIS SUMMARY")
    print("=" * 50)
    
    # Basic statistics
    print("\n1. Basic Statistics:")
    print(sentiment_matrix.describe().round(4))
    
    # Asset rankings by average sentiment
    print("\n2. Assets Ranked by Average Sentiment:")
    avg_sentiment = sentiment_matrix.mean().sort_values(ascending=False)
    for i, (asset, score) in enumerate(avg_sentiment.items(), 1):
        print(f"  {i}. {asset}: {score:.4f}")
    
    # Most volatile assets (by sentiment standard deviation)
    print("\n3. Most Volatile Assets (by sentiment):")
    volatility = sentiment_matrix.std().sort_values(ascending=False)
    for i, (asset, vol) in enumerate(volatility.items(), 1):
        print(f"  {i}. {asset}: {vol:.4f}")
    
    # Extreme sentiment periods
    print("\n4. Extreme Sentiment Periods:")
    overall_sentiment = sentiment_matrix.mean(axis=1)
    
    most_positive = overall_sentiment.idxmax()
    most_negative = overall_sentiment.idxmin()
    
    print(f"  Most positive period: {most_positive.strftime('%Y-%m')} (score: {overall_sentiment[most_positive]:.4f})")
    print(f"  Most negative period: {most_negative.strftime('%Y-%m')} (score: {overall_sentiment[most_negative]:.4f})")
    
    # Save summary statistics
    summary_stats = pd.DataFrame({
        'Mean_Sentiment': sentiment_matrix.mean(),
        'Std_Sentiment': sentiment_matrix.std(),
        'Min_Sentiment': sentiment_matrix.min(),
        'Max_Sentiment': sentiment_matrix.max()
    })
    
    summary_path = os.path.join(scraper.output_dir, "sentiment_summary_statistics.csv")
    summary_stats.to_csv(summary_path)
    print(f"\nSummary statistics saved to: {summary_path}")
else:
    print("No sentiment data available for summary statistics")


## Pipeline Complete

The sentiment analysis pipeline has been executed successfully. The system has:

1. **Scraped financial news** from Google News for multiple assets
2. **Analyzed sentiment** using FinBERT for each article  
3. **Calculated monthly sentiment scores** using the formula: (P_positive - P_negative) / N
4. **Generated a unified sentiment matrix** for all assets and time periods
5. **Created visualizations** to understand sentiment trends
6. **Produced summary statistics** for analysis insights

All results are saved in the `sentiment_results` directory for further analysis and integration with trading strategies.

### Key Features:
- **Modular design** with reusable utility functions
- **Rate limiting** to respect API constraints
- **Error handling** for robust operation
- **Clean visualizations** following KISS principle
- **Comprehensive output** in CSV format for easy integration
